In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ['HF_HOME'] = '/tmp/wendler/.hfcache'

In [ ]:
import torch
import sys
sys.path.append('../')
from SDLens import HookedStableDiffusionXLPipeline

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

dtype = torch.float32
    
pipe_inference = HookedStableDiffusionXLPipeline.from_pretrained("stabilityai/sdxl-turbo", 
                                                                 torch_dtype=dtype,
                                                                 device_map="balanced",
                                                                 variant=("fp16" if dtype==torch.float16 else None)
                                                                )

In [ ]:
if dtype == torch.float32:
    pipe_inference.text_encoder_2.to(dtype)

In [ ]:
import re
resnet_blocks = set()
conv_shortcut_set = set()
for n in pipe_inference.unet.named_modules():
    m = re.match(r"(.+resnets\.\d+)$", n[0])
    if m:
        block_name = m.group(1)
        resnet_blocks.add(block_name)
        for n2 in pipe_inference.unet.named_modules():
            if "conv_shortcut" in n2[0]:
                if block_name in n2[0]:
                    conv_shortcut_set.add(block_name.replace(".conv_shortcut",""))
print(resnet_blocks)
resnet_blocks_list = sorted(list(resnet_blocks))
print(resnet_blocks_list)

In [ ]:
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

prompt = "a photo of a colorful model"

conv_shortcut_list = sorted(list(conv_shortcut_set))
conv_shortcut_list = [f"unet.{name}.conv_shortcut" for name in conv_shortcut_list]
ref, cache = pipe_inference.run_with_cache(
                prompt,
                positions_to_cache=conv_shortcut_list,
                num_inference_steps=4,
                generator=torch.Generator(device=device).manual_seed(42),
                resolution=512,
                guidance_scale=0.0,
            )
print(cache['output']['unet.down_blocks.1.resnets.0.conv_shortcut'].shape)

In [ ]:
ref.images[0]

In [ ]:
from functools import partial
from utils import TimeDependentHook

def ablate_resnet(module, inputs, outputs, idx, surrogate_output=None):
    if surrogate_output is not None:
        return surrogate_output[:, idx]
    return inputs[0]

images = []
block_names_for_grid = []

for block_name in resnet_blocks_list:
    try: 
        if block_name in conv_shortcut_set:
            hook_fn = partial(ablate_resnet, surrogate_output=cache['output'][f"unet.{block_name}.conv_shortcut"])
            hook = TimeDependentHook(hook_fn, 4, apply_at_steps=[0, 1, 2, 3])
        else:
            hook = partial(ablate_resnet, idx=0)
        print(f"Ablating {block_name}")
        full_block_name = f'unet.{block_name}'

        out = pipe_inference.run_with_hooks(
            prompt,
            position_hook_dict={full_block_name: hook},
            num_inference_steps=4,
            generator=torch.Generator(device=device).manual_seed(42),
            guidance_scale=0.0,
        )
        images.append(out.images[0])
        block_names_for_grid.append(block_name)
    except:
        print(f"Failed to ablate {block_name}")

# Convert PIL images to tensors for make_grid
def pil_to_tensor(img):
    img_np = np.array(img)
    print(img_np.min(), img_np.max(), img_np.mean())
    return torch.from_numpy(np.array(img)).permute(2, 0, 1)

# Add the reference image as the first image in the grid
image_tensors = [pil_to_tensor(ref.images[0])] + [pil_to_tensor(img) for img in images]
labels = ["Reference"] + block_names_for_grid

n_images = len(image_tensors)
nrow = 4  # Adjust as needed
ncol = (n_images + nrow - 1) // nrow

# Plot with block names as titles
plt.figure(figsize=(4 * nrow, 4 * ncol))
for idx, (img_tensor, label) in enumerate(zip(image_tensors, labels)):
    plt.subplot(ncol, nrow, idx + 1)
    img = img_tensor.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
    plt.imshow(img)
    plt.title(label, fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()